In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Store Sales Forecasting (Kaggle) — Portfolio-Ready Project

## 0) 한 줄 요약(나중에 게시문장 1번으로 발전)
- Favorita(에콰도르) 매장×상품군 단위로 미래 판매량을 예측해 재고/발주/프로모션 운영 의사결정을 돕는다.

---

## 1) 문제정의 (Why)
- 목표: **(date, store_nbr, family)** 단위의 **미래 기간 판매량(sales)** 예측
- 비즈니스 맥락(현업 언어):
  - 예측 정확도 향상 → **과잉발주/폐기 리스크 감소**, **품절 리스크 감소**, **프로모션 물량 배분 최적화**

---

## 2) 지표 (Metric)
- Competition metric: **RMSLE**
- 운영/해석용 보조 지표(추가로 함께 기록 예정):
  - (예정) WMAPE 또는 SMAPE(전체/상위 매출 구간)

---

## 3) 결론(So What) 작성 규칙
- 매 단계(EDA/Feature/Model/Validation) 끝날 때마다 아래를 1~2문장으로 업데이트한다.
  1) 무엇을 확인/개선했는가?
  2) 그래서 운영 의사결정(발주/재고/프로모션)에 어떤 근거가 생겼는가?

---

## 4) 포트폴리오 기준(강제 체크리스트)
- 문제정의 → 지표 → 결론 흐름을 유지한다.
- 단순 그래프가 아니라 **왜 그런지(가설)** 를 함께 적는다.
- SQL/코드는 목적이 아니라 **의사결정 근거**로 사용한다.
- 결과는 **현업 언어(제품/마케팅/운영)** 로 번역한다.
- 내가 한 역할과 기여도를 매 버전마다 1문장으로 남긴다.

---

## 5) 실험 로그(버전 관리)
| Version | What changed | Hypothesis (Why) | Metric result | So What(의사결정 해석) | My role |
|---|---|---|---|---|---|
| v0 | family mean baseline (제출 성공) | 가장 단순한 기준선 확보 | LB:  | 기준선 대비 개선 방향 설정 | E2E 수행 |
| v1 |  |  |  |  |  |

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/competitions/store-sales-time-series-forecasting"

print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("Files:", os.listdir(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date"])
test  = pd.read_csv(f"{DATA_DIR}/test.csv",  parse_dates=["date"])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
oil = pd.read_csv(f"{DATA_DIR}/oil.csv", parse_dates=["date"])
holidays = pd.read_csv(f"{DATA_DIR}/holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv", parse_dates=["date"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("train:", train.shape, train["date"].min(), "~", train["date"].max())
print("test :", test.shape,  test["date"].min(),  "~", test["date"].max())

DATA_DIR exists: True
Files: ['oil.csv', 'sample_submission.csv', 'holidays_events.csv', 'stores.csv', 'train.csv', 'test.csv', 'transactions.csv']
train: (3000888, 6) 2013-01-01 00:00:00 ~ 2017-08-15 00:00:00
test : (28512, 5) 2017-08-16 00:00:00 ~ 2017-08-31 00:00:00


대회 데이터 7개 파일을 정상 로드했고, 학습기간(2013-01-01~2017-08-15)과 예측기간(2017-08-16~2017-08-31)이 명확히 분리된 시계열 예측 문제임을 확인

In [2]:
# [셀 3] 예측 단위(키) 검증: (date, store_nbr, family) 1행=1예측대상인지 확인

key_cols = ["date", "store_nbr", "family"]

print("n_store(train):", train["store_nbr"].nunique())
print("n_family(train):", train["family"].nunique())

print("n_store(test):", test["store_nbr"].nunique())
print("n_family(test):", test["family"].nunique())

print("train unique key rows:", train[key_cols].drop_duplicates().shape[0], "/", train.shape[0])
print("test  unique key rows:", test[key_cols].drop_duplicates().shape[0], "/", test.shape[0])

# test가 며칠치인지(양끝 포함) 계산
n_days_test = (test["date"].max() - test["date"].min()).days + 1
print("test days (inclusive):", n_days_test)

# 행 수가 일수*매장수*상품군수와 맞는지 검증
expected = n_days_test * test["store_nbr"].nunique() * test["family"].nunique()
print("expected rows:", expected, "| actual rows:", len(test))

n_store(train): 54
n_family(train): 33
n_store(test): 54
n_family(test): 33
train unique key rows: 3000888 / 3000888
test  unique key rows: 28512 / 28512
test days (inclusive): 16
expected rows: 28512 | actual rows: 28512


**> “예측 단위가 (date, store_nbr, family)로 완전 유니크하며, test는 16일×54매장×33상품군=28,512행으로 ‘미래 16일 판매 예측’ 문제임을 확정했다.”**

In [3]:
# [셀 4] sales 분포 요약: 0 비율 + 분위수 + 기술통계
zero_ratio = (train["sales"] == 0).mean()
print("sales==0 ratio:", zero_ratio)

q = train["sales"].quantile([0, .25, .5, .75, .9, .95, .99, .999, 1.0])
print("\nquantiles:\n", q)

print("\ndescribe:\n", train["sales"].describe())

sales==0 ratio: 0.3129506999261552

quantiles:
 0.000         0.000000
0.250         0.000000
0.500        11.000000
0.750       195.847250
0.900       867.000000
0.950      1965.000000
0.990      5507.000000
0.999     12076.225548
1.000    124717.000000
Name: sales, dtype: float64

describe:
 count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64


**“sales는 0이 약 31.3%로 많고(25%도 0), 분포가 극단적으로 오른쪽으로 치우쳐(max 124,717) 로그 기반 오차지표(RMSLE) 및 log1p 변환이 합리적인 시계열 수요예측 문제임을 확인했다.”**> 

In [4]:
# [셀 5] onpromotion이 sales와 같이 움직이는지: 상관 + 구간별 평균

# 전체 상관(대략 힌트)
corr = train[["onpromotion","sales"]].corr().iloc[0,1]
print("corr(onpromotion, sales) overall:", corr)

# onpromotion=0 vs >0 비교(가장 직관적인 2그룹)
mean0 = train.loc[train["onpromotion"]==0, "sales"].mean()
mean1 = train.loc[train["onpromotion"]>0, "sales"].mean()
print("mean sales when onpromotion==0 :", mean0)
print("mean sales when onpromotion>0  :", mean1)

# 구간(bin)으로 묶어서 평균 sales 비교(노이즈 줄이기)
bins = [-1,0,1,2,3,5,10,20,50,100,1000]
promo_bin = pd.cut(train["onpromotion"], bins=bins)
bin_mean = train.groupby(promo_bin, observed=False)["sales"].mean()
print("\nmean sales by promo_bin:\n", bin_mean)

corr(onpromotion, sales) overall: 0.4279232048121194
mean sales when onpromotion==0 : 158.2466813936427
mean sales when onpromotion>0  : 1137.6937303133943

mean sales by promo_bin:
 onpromotion
(-1, 0]         158.246681
(0, 1]          467.556532
(1, 2]          662.925632
(2, 3]          871.408092
(3, 5]          987.707384
(5, 10]        1133.186482
(10, 20]       1403.325230
(20, 50]       2167.008534
(50, 100]      3300.882681
(100, 1000]    4275.335174
Name: sales, dtype: float64


**“프로모션 품목 수(onpromotion)는 sales와 중간 정도 양의 상관(0.428)을 보였고, 프로모션이 있는 날의 평균 판매가(≈1138)가 없는 날(≈158)보다 크게 높아 ‘프로모션 변수는 판매 예측과 운영(프로모션 물량/재고 배분) 의사결정에 핵심 신호’임을 확인했다.”**


1. 상관 0.428은 “완벽”은 아니지만, 명확히 방향성이 있는 신호
2. onpromotion==0 평균 158 vs >0 평균 1138 → 7배 이상 차이
3. bin별 평균도 onpromotion 구간이 커질수록 sales가 꾸준히 상승→ “프로모션 강도”가 판매를 끌어올리는 패턴이 데이터에서 관찰됨


**왜(목적)**: 전체 300만 행을 전부 조인하면 시간이 오래 걸리고, 잘못된 키 조인(중복/카티전) 시 원인 추적이 어렵다.


따라서 **조인 품질(커버리지/결측/키 불일치)**을 빠르게 검증하기 위해, 전체 행이 아닌 ‘조인 키의 유니크 조합’만 추출해 전수에 가깝게 점검한다.


**무엇을 확인하려고?**

stores: store_nbr 기준으로 메타가 100% 붙는지

oil: 날짜(date) 기준으로 값이 얼마나 비는지(train/test 모두)

transactions: (date, store_nbr) 기준 결측이 있는지

holidays: 우선 “해당 날짜에 이벤트가 존재하는지”만(지역/transfer 처리는 다음 단계에서 설계)

In [5]:
# [셀 6] 전수형(키 유니크 기반) 조인 커버리지 점검

import pandas as pd

# 1) stores: store_nbr 유니크 54개 전수 점검
k_store = train[["store_nbr"]].drop_duplicates()
m_store = k_store.merge(stores, on="store_nbr", how="left")
print("[stores] missing any store metadata?:", m_store.isna().any(axis=1).mean())

# 2) oil: train/test 날짜 전수(유니크 date) 기준으로 결측 점검
k_date_train = train[["date"]].drop_duplicates()
k_date_test  = test[["date"]].drop_duplicates()

m_oil_train = k_date_train.merge(oil, on="date", how="left")
m_oil_test  = k_date_test.merge(oil, on="date", how="left")

print("[oil] train date coverage missing ratio:", m_oil_train["dcoilwtico"].isna().mean())
print("[oil] test  date coverage missing ratio:", m_oil_test["dcoilwtico"].isna().mean())

# 3) transactions: (date, store_nbr) 유니크 기준 전수 점검 (train에만 존재)
k_ds = train[["date","store_nbr"]].drop_duplicates()
m_tx = k_ds.merge(transactions, on=["date","store_nbr"], how="left")
print("[transactions] train (date,store) missing ratio:", m_tx["transactions"].isna().mean())

# 4) holidays: date별 이벤트 존재 여부(단순 커버리지)만 우선 확인
holiday_by_date = holidays.groupby("date").size().rename("n_events").reset_index()
m_h_train = k_date_train.merge(holiday_by_date, on="date", how="left")
m_h_test  = k_date_test.merge(holiday_by_date, on="date", how="left")

print("[holidays] train dates with any event ratio:", (m_h_train["n_events"].fillna(0) > 0).mean())
print("[holidays] test  dates with any event ratio:", (m_h_test["n_events"].fillna(0) > 0).mean())

[stores] missing any store metadata?: 0.0
[oil] train date coverage missing ratio: 0.30938242280285033
[oil] test  date coverage missing ratio: 0.25
[transactions] train (date,store) missing ratio: 0.08190375648807953
[holidays] train dates with any event ratio: 0.1496437054631829
[holidays] test  dates with any event ratio: 0.0625


“stores 메타는 결측 없이 100% 매칭 

oil은 날짜 기준 결측이 크며(train 30.9%, test 25%) 

transactions도 (date,store) 기준 약 8.2% 결측이 있어 전체 피처 테이블 구축 전에 결측 처리/조인 전략을 설계해야 함을 확인

**왜(목적)**
셀6에서 oil 결측이 크게 보였는데(train 30.9%, test 25%),
이 결측이 

(1) oil.csv에 해당 날짜가 아예 없어서(키 불일치)인지,

(2) 날짜는 있는데 값이 NaN이라서(값 결측)인지 구분해야 한다.


원인이 다르면 처리 전략도 달라짐:
키 불일치면 date 정규화/타입 정리가 먼저
값 NaN이면 ffill/보간 등 결측 처리가 먼저

In [6]:
# [셀 7] oil 결측 원인 분해: 키(date) 문제 vs 값(dcoilwtico) NaN 문제

# (1) oil.csv 자체 NaN 비율(파일 안에 값이 비어있는 정도)
print("oil.csv NaN ratio (dcoilwtico):", oil["dcoilwtico"].isna().mean())

# (2) train/test의 날짜가 oil.csv에 '존재'하는지(키 커버리지)
train_dates = train["date"].drop_duplicates()
test_dates  = test["date"].drop_duplicates()
oil_dates   = oil["date"].drop_duplicates()

print("train dates exist in oil.csv ratio:", train_dates.isin(oil_dates).mean())
print("test  dates exist in oil.csv ratio:", test_dates.isin(oil_dates).mean())

# (3) oil.csv에 날짜는 있는데 값이 NaN인 날짜 비율(키는 맞는데 값이 없는 케이스)
oil_nan_dates = oil.loc[oil["dcoilwtico"].isna(), "date"].drop_duplicates()

print("train dates with oil value NaN ratio:", train_dates.isin(oil_nan_dates).mean())
print("test  dates with oil value NaN ratio:", test_dates.isin(oil_nan_dates).mean())

# (4) 참고: oil.csv에 날짜 자체가 없는 train/test 날짜 개수(정확히 몇 일인지)
missing_train_dates = train_dates[~train_dates.isin(oil_dates)]
missing_test_dates  = test_dates[~test_dates.isin(oil_dates)]

print("n_train_dates_not_in_oil:", len(missing_train_dates))
print("n_test_dates_not_in_oil :", len(missing_test_dates))

if len(missing_train_dates) > 0:
    print("example missing train dates:", missing_train_dates.sort_values().head(5).tolist())
if len(missing_test_dates) > 0:
    print("example missing test dates :", missing_test_dates.sort_values().head(5).tolist())

oil.csv NaN ratio (dcoilwtico): 0.035303776683087026
train dates exist in oil.csv ratio: 0.7143705463182898
test  dates exist in oil.csv ratio: 0.75
train dates with oil value NaN ratio: 0.023752969121140142
test  dates with oil value NaN ratio: 0.0
n_train_dates_not_in_oil: 481
n_test_dates_not_in_oil : 4
example missing train dates: [Timestamp('2013-01-05 00:00:00'), Timestamp('2013-01-06 00:00:00'), Timestamp('2013-01-12 00:00:00'), Timestamp('2013-01-13 00:00:00'), Timestamp('2013-01-19 00:00:00')]
example missing test dates : [Timestamp('2017-08-19 00:00:00'), Timestamp('2017-08-20 00:00:00'), Timestamp('2017-08-26 00:00:00'), Timestamp('2017-08-27 00:00:00')]


“oil 결측의 주된 원인은 값 NaN(3.5%)이 아니라 ‘주말 등 특정 날짜가 oil.csv에 아예 존재하지 않는 키 불일치(Train 481일, Test 4일)’로 확인되어, oil 피처는 날짜 커버리지를 확보하기 위해 전일값 기반 보간(ffill)로 일 단위 연속 시계열로 보정하는 전략이 타당하다.”

**왜(목적)**

oil.csv는 주말 등 일부 날짜가 빠져 있어 조인 시 결측이 크게 발생한다.

판매 예측은 일 단위이므로 oil도 일 단위 연속 시계열로 만들어야 한다.

가장 보수적인 방법은 전일값 유지(ffill): “주말에는 직전 영업일 가격이 유지됐다”로 가정.

In [7]:
# [셀 8] oil을 일 단위로 연속화 + ffill/bfill로 결측 보정

import pandas as pd

# (1) train/test 전체 기간의 '모든 날짜'를 일 단위로 생성
full_dates = pd.date_range(
    start=min(train["date"].min(), test["date"].min()),
    end=max(train["date"].max(), test["date"].max()),
    freq="D"
)

# (2) oil을 date를 인덱스로 바꾸고(full_dates 기준) 재색인해서
#     oil에 없는 날짜(주말 등)를 NaN으로 만들기
oil_daily = (
    oil.set_index("date")
       .reindex(full_dates)
       .rename_axis("date")
       .reset_index()
)

# (3) 결측을 전일 값으로 채움 (forward fill)
oil_daily["dcoilwtico_filled"] = oil_daily["dcoilwtico"].ffill()

# (4) 맨 앞부분이 NaN인 경우(시작일에 값이 없을 때)만 뒤 값으로 한 번 더 채움 (backward fill)
oil_daily["dcoilwtico_filled"] = oil_daily["dcoilwtico_filled"].bfill()

# (5) 품질 체크: 결측이 남아있는지, train/test 날짜가 전부 커버되는지 확인
print("oil_daily date range:", oil_daily["date"].min(), "~", oil_daily["date"].max())
print("missing ratio after fill:", oil_daily["dcoilwtico_filled"].isna().mean())

train_dates = train["date"].drop_duplicates()
test_dates  = test["date"].drop_duplicates()
oil_daily_dates = oil_daily["date"].drop_duplicates()

print("train date coverage:", train_dates.isin(oil_daily_dates).mean())
print("test  date coverage:", test_dates.isin(oil_daily_dates).mean())

oil_daily.head(3)

oil_daily date range: 2013-01-01 00:00:00 ~ 2017-08-31 00:00:00
missing ratio after fill: 0.0
train date coverage: 1.0
test  date coverage: 1.0


,date,dcoilwtico,dcoilwtico_filled
0,2013-01-01,NaN,93.14
1,2013-01-02,93.14,93.14
2,2013-01-03,92.97,92.97


“주말 등 oil 데이터가 없는 날짜를 포함해 2013-01-01~2017-08-31 전 기간을 일 단위로 연속화하고 ffill/bfill로 결측을 0%로 만들어, train/test 모든 날짜(커버리지 1.0)에 대해 oil 피처를 안정적으로 조인 가능한 상태로 정제했다.”

왜(목적)
셀6에서 (date, store_nbr) 기준 transactions 결측이 약 8.2%로 확인됐다.

결측이 생기는 원인을 알아야 한다:

-특정 매장만 누락인지?

-특정 기간(초기/특정 연도) 집중인지?

-특정 요일(휴무일 등) 관련인지?

또한 transactions는 test에 존재하지 않으므로, 향후 “어떻게 활용할지(사용/미사용/대체 피처)” 결정을 위한 근거가 필요하다.

In [8]:
# [셀 9] transactions 결측 원인 진단: 어떤 store/기간에서 빠지는지 확인

# (1) train의 (date, store) 유니크 조합을 기준으로 transactions 커버리지 확인
k_ds = train[["date","store_nbr"]].drop_duplicates()
m_tx = k_ds.merge(transactions, on=["date","store_nbr"], how="left")

print("overall missing ratio:", m_tx["transactions"].isna().mean())

# (2) store별 결측 비율(어떤 매장이 특히 누락되는지)
store_missing = (m_tx.assign(missing=m_tx["transactions"].isna())
                    .groupby("store_nbr")["missing"]
                    .mean()
                    .sort_values(ascending=False))
print("\nstore missing ratio (top 10):")
print(store_missing.head(10))

# (3) 날짜별 결측 비율(특정 기간에 집중되는지)
date_missing = (m_tx.assign(missing=m_tx["transactions"].isna())
                   .groupby("date")["missing"]
                   .mean())
print("\nmax date missing ratio:", date_missing.max())
print("dates with missing ratio > 0.5 (count):", (date_missing > 0.5).sum())

# (4) 결측이 많은 상위 날짜 몇 개 확인
top_missing_dates = date_missing.sort_values(ascending=False).head(10)
print("\ntop 10 missing dates:")
print(top_missing_dates)

overall missing ratio: 0.08190375648807953

store missing ratio (top 10):
store_nbr
52    0.929929
22    0.601544
42    0.572447
21    0.555819
29    0.480998
20    0.460214
53    0.307007
36    0.078979
18    0.070071
24    0.063539
Name: missing, dtype: float64

max date missing ratio: 1.0
dates with missing ratio > 0.5 (count): 7

top 10 missing dates:
date
2016-01-01    1.000000
2016-01-03    1.000000
2017-01-01    0.981481
2015-01-01    0.981481
2013-01-01    0.981481
2014-01-01    0.962963
2016-01-04    0.740741
2016-01-02    0.333333
2013-06-19    0.203704
2013-01-31    0.148148
Name: missing, dtype: float64


“transactions는 전체적으로 8.19% 결측이지만, 특정 매장(52번은 93%, 21~22~42 등도 50%+)에 결측이 집중되고, 1/1 전후(특히 2013~2017-01-01 등 특정 날짜)에는 대부분 매장이 누락되는 패턴이 확인되어, 단순 평균 대체보다는 ‘매장별 커버리지 기준 처리 + 특정 날짜(휴일) 처리’ 전략이 필요하다.”

**transactions를 “미래에도 쓸 수 있게” 변환 (expected_transactions)**
1) 왜(목적)
transactions는 수요예측에서 중요한 “매장 유입(traffic)” 신호지만, test에는 없어서 그대로는 feature로 쓸 수 없다.

따라서 train의 transactions로부터 미래 16일에도 계산 가능한 트래픽 추정치(expected_transactions)를 만들어 sales 모델에 포함한다.

가장 단순/설명 가능한 방식:
(최근 수준) × (요일 패턴 factor)

In [9]:
# [셀10] expected_transactions 생성: (최근 28일 수준) × (store별 요일 패턴)

import pandas as pd
import numpy as np

TRAIN_END = train["date"].max()          # 2017-08-15
TEST_START = test["date"].min()          # 2017-08-16
TEST_END = test["date"].max()            # 2017-08-31

# 1) transactions에 요일(dow) 추가
tx = transactions.copy()
tx["dow"] = tx["date"].dt.dayofweek  # 0=Mon ... 6=Sun

# 2) store별 전체 평균 거래량(기준)
store_mean = tx.groupby("store_nbr")["transactions"].mean()

# 3) store×요일 평균 거래량 → 요일 factor(=요일 평균 / store 평균)
dow_mean = tx.groupby(["store_nbr","dow"])["transactions"].mean()
dow_factor = (dow_mean / store_mean).replace([np.inf, -np.inf], np.nan).fillna(1.0)

# 4) 최근 수준(level): train 마지막 28일 평균 거래량(매장별)
level_window = 28
level_start = TRAIN_END - pd.Timedelta(days=level_window)
recent_level = (tx[tx["date"] > level_start]
                .groupby("store_nbr")["transactions"]
                .mean())

# 5) test의 모든 (date, store_nbr)에 대해 expected_transactions 계산
test_dates = pd.DataFrame({"date": pd.date_range(TEST_START, TEST_END, freq="D")})
test_dates["dow"] = test_dates["date"].dt.dayofweek

store_list = pd.DataFrame({"store_nbr": sorted(train["store_nbr"].unique())})
grid = test_dates.merge(store_list, how="cross")

# level과 factor 매핑
grid["level"] = grid["store_nbr"].map(recent_level)
# recent_level이 없는 store(드물게 가능) 대비: 전체 store_mean으로 대체
grid["level"] = grid["level"].fillna(grid["store_nbr"].map(store_mean))

grid["factor"] = grid.apply(lambda r: dow_factor.get((r["store_nbr"], r["dow"]), 1.0), axis=1)

grid["expected_transactions"] = grid["level"] * grid["factor"]

print("grid shape:", grid.shape)  # 16일 * 54매장 = 864
grid.head(3)

grid shape: (864, 6)


,date,dow,store_nbr,level,factor,expected_transactions
0,2017-08-16,2,1,1479.928571,1.227673,1816.868308
1,2017-08-16,2,2,1793.571429,1.016987,1824.039468
2,2017-08-16,2,3,3122.535714,0.981512,3064.805183


“test 기간(16일×54매장)에 대해, 과거 transactions로부터 요일 패턴과 최근 수준을 반영한 expected_transactions를 생성해 ‘미래에도 사용 가능한 트래픽 피처’를 확보했다.”

) 왜(목적)
셀10에서 만든 expected_transactions = 최근수준(level) × 요일패턴(factor)가
“그럴듯하다”고 말하려면 근거(검증 점수)가 필요.

그래서 test 기간을 그대로 쓸 수 없으니, train의 마지막 16일을 “가짜 test”로 두고:
그 이전 데이터로 factor/level을 만들고
마지막 16일의 transactions를 예측해본 뒤
실제 transactions와 비교해서 오차를 계산한다.

이 검증이 통과되면, expected_transactions를 sales 모델에 넣는 논리가 훨씬 설득력 있어져.

In [10]:
# [셀 11] expected_transactions 백테스트 검증 (마지막 16일을 가짜 test로)

import pandas as pd
import numpy as np

# 1) holdout(검증) 구간 = transactions가 존재하는 마지막 날짜 기준 16일
TX_END = transactions["date"].max()                 # 보통 2017-08-15
HOLDOUT_DAYS = 16
VAL_START = TX_END - pd.Timedelta(days=HOLDOUT_DAYS-1)  # 양끝 포함 16일
VAL_END = TX_END

# 2) 학습 구간: VAL_START 이전까지만 사용해서 패턴/수준을 계산
tx = transactions.copy()
tx["dow"] = tx["date"].dt.dayofweek

tx_hist = tx[tx["date"] < VAL_START].copy()   # 과거(학습) 구간
tx_val  = tx[(tx["date"] >= VAL_START) & (tx["date"] <= VAL_END)].copy()  # 검증 구간

print("TX_END:", TX_END)
print("VAL period:", VAL_START.date(), "~", VAL_END.date(), "| days:", HOLDOUT_DAYS)
print("tx_hist range:", tx_hist["date"].min().date(), "~", tx_hist["date"].max().date())

# 3) store_mean / dow_factor (과거 구간으로만 계산)
store_mean = tx_hist.groupby("store_nbr")["transactions"].mean()
dow_mean = tx_hist.groupby(["store_nbr","dow"])["transactions"].mean()
dow_factor = dow_mean.div(store_mean, level="store_nbr").replace([np.inf, -np.inf], np.nan).fillna(1.0)

# 4) recent_level: 검증 시작 직전 28일 평균(과거 구간에서만)
level_window = 28
level_start = VAL_START - pd.Timedelta(days=level_window)
recent_level = (tx_hist[tx_hist["date"] > level_start]
                .groupby("store_nbr")["transactions"]
                .mean())

# 5) 검증 기간(16일) 날짜×매장 grid 생성
val_dates = pd.DataFrame({"date": pd.date_range(VAL_START, VAL_END, freq="D")})
val_dates["dow"] = val_dates["date"].dt.dayofweek
store_list = pd.DataFrame({"store_nbr": sorted(tx["store_nbr"].unique())})

try:
    grid_val = val_dates.merge(store_list, how="cross")
except TypeError:
    val_dates["_k"] = 1
    store_list["_k"] = 1
    grid_val = val_dates.merge(store_list, on="_k").drop(columns=["_k"])

# 6) level + factor 매핑 후 예측치 생성
grid_val["level"] = grid_val["store_nbr"].map(recent_level)
grid_val["level"] = grid_val["level"].fillna(grid_val["store_nbr"].map(store_mean))

grid_val["factor"] = grid_val.apply(
    lambda r: dow_factor.get((r["store_nbr"], r["dow"]), 1.0),
    axis=1
)

grid_val["pred_tx"] = grid_val["level"] * grid_val["factor"]

# 7) 실제 transactions 붙여서 오차 계산
actual = tx_val[["date","store_nbr","transactions"]].rename(columns={"transactions":"actual_tx"})
eval_df = grid_val.merge(actual, on=["date","store_nbr"], how="left")

# 실제가 없는 곳(원래 결측)은 평가에서 제외 (공정한 비교)
eval_df = eval_df.dropna(subset=["actual_tx"]).copy()

# 오차 지표: MAE, WMAPE(가중치 절대오차율)
eval_df["abs_err"] = (eval_df["pred_tx"] - eval_df["actual_tx"]).abs()

mae = eval_df["abs_err"].mean()
wmape = eval_df["abs_err"].sum() / eval_df["actual_tx"].sum()

print("Eval rows:", len(eval_df))
print("MAE:", mae)
print("WMAPE:", wmape)

# 8) (선택) 어떤 매장이 특히 안 맞는지 Top5
store_wmape = (eval_df.groupby("store_nbr")
                      .apply(lambda g: g["abs_err"].sum() / g["actual_tx"].sum())
                      .sort_values(ascending=False)
                      .head(5))
print("\nWorst 5 stores by WMAPE:\n", store_wmape)

TX_END: 2017-08-15 00:00:00
VAL period: 2017-07-31 ~ 2017-08-15 | days: 16
tx_hist range: 2013-01-01 ~ 2017-07-30
Eval rows: 864
MAE: 116.19509399326711
WMAPE: 0.0704862869511248

Worst 5 stores by WMAPE:
 store_nbr
25    0.222081
26    0.110803
9     0.107534
48    0.105889
42    0.104018
dtype: float64


/tmp/ipykernel_55/3406558245.py:77: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["abs_err"].sum() / g["actual_tx"].sum())
